# AcademyFlow - Improved ML Model Training
### Advanced Ensemble with Feature Engineering, Hyperparameter Tuning & Stacking

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import (
    RandomForestRegressor, GradientBoostingRegressor,
    AdaBoostRegressor, StackingRegressor, VotingRegressor
)
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
import joblib
import warnings
warnings.filterwarnings('ignore')

print('All imports loaded successfully.')

In [ ]:
# Load and explore dataset
df = pd.read_csv('Student_Performance.csv')
print(f'Dataset shape: {df.shape}')
print(f'\nMissing values:\n{df.isnull().sum()}')
print(f'\nBasic stats:')
df.describe()

In [ ]:
# Encode categorical
df['Extracurricular Activities'] = (df['Extracurricular Activities'] == 'Yes').astype(int)

# ===== ADVANCED FEATURE ENGINEERING =====

# Original engineered features (backward compatible)
df['Study_Sleep_Ratio'] = df['Hours Studied'] / (df['Sleep Hours'] + 1)
df['Total_Effort'] = df['Hours Studied'] + df['Sample Question Papers Practiced']
df['Previous_Score_Normalized'] = df['Previous Scores'] / 100
df['Sleep_Efficiency'] = df['Sleep Hours'] * df['Previous Scores'] / 100

# NEW: Interaction features
df['Study_Score_Interaction'] = df['Hours Studied'] * df['Previous Scores'] / 100
df['Effort_Efficiency'] = df['Total_Effort'] * df['Previous_Score_Normalized']
df['Study_Squared'] = df['Hours Studied'] ** 2
df['Score_Squared'] = (df['Previous Scores'] / 100) ** 2
df['Papers_Per_Hour'] = df['Sample Question Papers Practiced'] / (df['Hours Studied'] + 1)
df['Balanced_Lifestyle'] = (
    df['Sleep_Efficiency'] * df['Extracurricular Activities'].replace(0, 0.5)
)

print(f'Features after engineering: {df.shape[1] - 1}')
print(f'Columns: {list(df.columns)}')

In [ ]:
# Correlation analysis with target
corr_with_target = df.corr()['Performance Index'].drop('Performance Index').sort_values(ascending=False)
print('Feature correlation with Performance Index:')
print(corr_with_target.round(4))

plt.figure(figsize=(10, 6))
corr_with_target.plot(kind='barh', color=['#6366F1' if v > 0 else '#EF4444' for v in corr_with_target])
plt.title('Feature Correlation with Performance', fontsize=14, fontweight='bold')
plt.xlabel('Correlation')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Prepare data
X = df.drop('Performance Index', axis=1)
y = df['Performance Index']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features for distance-based models
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# ===== TRAIN & COMPARE MANY MODELS =====

kfold = KFold(n_splits=10, shuffle=True, random_state=42)

models = {
    'Linear Regression': LinearRegression(),
    'Ridge (alpha=1)': Ridge(alpha=1.0),
    'Ridge (alpha=10)': Ridge(alpha=10.0),
    'Lasso': Lasso(alpha=0.1),
    'ElasticNet': ElasticNet(alpha=0.1, l1_ratio=0.5),
    'KNN (k=5)': KNeighborsRegressor(n_neighbors=5),
    'KNN (k=10)': KNeighborsRegressor(n_neighbors=10),
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=12, min_samples_leaf=3, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=300, max_depth=5, learning_rate=0.1,
        subsample=0.8, min_samples_leaf=5, random_state=42
    ),
    'AdaBoost': AdaBoostRegressor(n_estimators=200, learning_rate=0.05, random_state=42),
}

results = {}
trained_models = {}

for name, model in models.items():
    # Use scaled data for KNN, unscaled for tree-based
    use_scaled = 'KNN' in name
    Xtr = X_train_scaled if use_scaled else X_train
    Xte = X_test_scaled if use_scaled else X_test
    Xfull = scaler.transform(X) if use_scaled else X
    
    model.fit(Xtr, y_train)
    y_pred = model.predict(Xte)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    cv = cross_val_score(model, Xfull, y, cv=kfold, scoring='r2')
    
    results[name] = {
        'MAE': mae, 'RMSE': rmse, 'R2': r2,
        'CV_R2_Mean': cv.mean(), 'CV_R2_Std': cv.std()
    }
    trained_models[name] = model
    
    print(f'{name:25s} | R²: {r2:.4f} | MAE: {mae:.3f} | RMSE: {rmse:.3f} | CV R²: {cv.mean():.4f} ± {cv.std():.4f}')

In [ ]:
# ===== HYPERPARAMETER TUNING for Gradient Boosting =====

param_grid = {
    'n_estimators': [200, 300, 500],
    'max_depth': [4, 5, 6],
    'learning_rate': [0.05, 0.1, 0.15],
    'subsample': [0.8, 0.9],
    'min_samples_leaf': [3, 5],
}

print('Running GridSearchCV for Gradient Boosting (this may take a few minutes)...')
gb_grid = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    param_grid, cv=5, scoring='r2', n_jobs=-1, verbose=0
)
gb_grid.fit(X_train, y_train)

print(f'\nBest params: {gb_grid.best_params_}')
print(f'Best CV R²: {gb_grid.best_score_:.4f}')

# Evaluate tuned model
gb_tuned = gb_grid.best_estimator_
y_pred_tuned = gb_tuned.predict(X_test)
r2_tuned = r2_score(y_test, y_pred_tuned)
mae_tuned = mean_absolute_error(y_test, y_pred_tuned)
rmse_tuned = np.sqrt(mean_squared_error(y_test, y_pred_tuned))

print(f'\nTuned GB Test Performance:')
print(f'R²: {r2_tuned:.4f} | MAE: {mae_tuned:.3f} | RMSE: {rmse_tuned:.3f}')

trained_models['GB Tuned'] = gb_tuned
results['GB Tuned'] = {
    'MAE': mae_tuned, 'RMSE': rmse_tuned, 'R2': r2_tuned,
    'CV_R2_Mean': gb_grid.best_score_, 'CV_R2_Std': 0
}

In [ ]:
# ===== STACKING ENSEMBLE =====

estimators = [
    ('ridge', Ridge(alpha=1.0)),
    ('rf', RandomForestRegressor(n_estimators=200, max_depth=12, min_samples_leaf=3, random_state=42)),
    ('gb', gb_tuned),
]

stacking_model = StackingRegressor(
    estimators=estimators,
    final_estimator=Ridge(alpha=0.5),
    cv=5,
    n_jobs=-1,
)

print('Training Stacking Ensemble...')
stacking_model.fit(X_train, y_train)
y_pred_stack = stacking_model.predict(X_test)

r2_stack = r2_score(y_test, y_pred_stack)
mae_stack = mean_absolute_error(y_test, y_pred_stack)
rmse_stack = np.sqrt(mean_squared_error(y_test, y_pred_stack))
cv_stack = cross_val_score(stacking_model, X, y, cv=5, scoring='r2')

print(f'\nStacking Ensemble Performance:')
print(f'R²: {r2_stack:.4f} | MAE: {mae_stack:.3f} | RMSE: {rmse_stack:.3f}')
print(f'CV R²: {cv_stack.mean():.4f} ± {cv_stack.std():.4f}')

trained_models['Stacking Ensemble'] = stacking_model
results['Stacking Ensemble'] = {
    'MAE': mae_stack, 'RMSE': rmse_stack, 'R2': r2_stack,
    'CV_R2_Mean': cv_stack.mean(), 'CV_R2_Std': cv_stack.std()
}

In [ ]:
# ===== VOTING ENSEMBLE =====

voting_model = VotingRegressor(
    estimators=[
        ('lr', LinearRegression()),
        ('ridge', Ridge(alpha=1.0)),
        ('rf', RandomForestRegressor(n_estimators=200, max_depth=12, min_samples_leaf=3, random_state=42)),
        ('gb', gb_tuned),
    ],
    n_jobs=-1,
)

print('Training Voting Ensemble...')
voting_model.fit(X_train, y_train)
y_pred_vote = voting_model.predict(X_test)

r2_vote = r2_score(y_test, y_pred_vote)
mae_vote = mean_absolute_error(y_test, y_pred_vote)
rmse_vote = np.sqrt(mean_squared_error(y_test, y_pred_vote))

print(f'\nVoting Ensemble Performance:')
print(f'R²: {r2_vote:.4f} | MAE: {mae_vote:.3f} | RMSE: {rmse_vote:.3f}')

trained_models['Voting Ensemble'] = voting_model
results['Voting Ensemble'] = {
    'MAE': mae_vote, 'RMSE': rmse_vote, 'R2': r2_vote,
    'CV_R2_Mean': 0, 'CV_R2_Std': 0
}

In [ ]:
# ===== FINAL COMPARISON =====

results_df = pd.DataFrame(results).T.sort_values('R2', ascending=False)
print('\n' + '='*80)
print('FINAL MODEL COMPARISON (sorted by R²)')
print('='*80)
print(results_df[['R2', 'MAE', 'RMSE', 'CV_R2_Mean']].round(4))

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
colors = ['#6366F1' if i == 0 else '#8B5CF6' if i < 3 else '#94A3B8' for i in range(len(results_df))]

for idx, metric in enumerate(['R2', 'MAE', 'RMSE']):
    results_df[metric].plot(kind='barh', ax=axes[idx], color=colors)
    axes[idx].set_title(metric, fontsize=14, fontweight='bold')
    axes[idx].grid(alpha=0.3)

plt.suptitle('Model Performance Comparison', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ===== SELECT & SAVE BEST MODEL =====

best_name = results_df['R2'].idxmax()
best_model = trained_models[best_name]

print(f'Best Model: {best_name}')
print(f'R²: {results_df.loc[best_name, "R2"]:.4f}')
print(f'MAE: {results_df.loc[best_name, "MAE"]:.3f}')
print(f'RMSE: {results_df.loc[best_name, "RMSE"]:.3f}')

# Feature names must match what the backend produces.
# The backend constructs 9 features (5 raw + 4 engineered).
# Our improved model uses 15 features (5 raw + 10 engineered).
# We need to save feature_names so the backend can reconstruct them.

model_package = {
    'best_model': best_model,
    'best_model_name': best_name,
    'all_models': trained_models,
    'feature_names': list(X.columns),
    'results': results,
    'scaler': scaler,
    'version': '2.0',
}

joblib.dump(model_package, 'enhanced_model.pkl')
print(f'\nModel saved as enhanced_model.pkl')
print(f'Features ({len(X.columns)}): {list(X.columns)}')

In [ ]:
# ===== FEATURE IMPORTANCE (from best tree-based model) =====

# Use gradient boosting for importance analysis
if hasattr(gb_tuned, 'feature_importances_'):
    fi = pd.DataFrame({
        'Feature': X.columns,
        'Importance': gb_tuned.feature_importances_
    }).sort_values('Importance', ascending=True)
    
    plt.figure(figsize=(10, 7))
    plt.barh(fi['Feature'], fi['Importance'], color='#6366F1')
    plt.xlabel('Importance')
    plt.title('Feature Importance (Tuned Gradient Boosting)', fontsize=14, fontweight='bold')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# ===== RESIDUAL ANALYSIS =====

y_pred_best = best_model.predict(X_test)
residuals = y_test - y_pred_best

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Predicted vs Actual
axes[0].scatter(y_test, y_pred_best, alpha=0.3, s=10, color='#6366F1')
axes[0].plot([0, 100], [0, 100], 'r--', linewidth=1)
axes[0].set_xlabel('Actual')
axes[0].set_ylabel('Predicted')
axes[0].set_title('Predicted vs Actual', fontweight='bold')
axes[0].grid(alpha=0.3)

# Residual distribution
axes[1].hist(residuals, bins=50, color='#8B5CF6', alpha=0.7)
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Residual Distribution', fontweight='bold')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].grid(alpha=0.3)

# Residual vs Predicted
axes[2].scatter(y_pred_best, residuals, alpha=0.3, s=10, color='#10B981')
axes[2].axhline(0, color='red', linestyle='--')
axes[2].set_xlabel('Predicted')
axes[2].set_ylabel('Residual')
axes[2].set_title('Residual vs Predicted', fontweight='bold')
axes[2].grid(alpha=0.3)

plt.suptitle(f'Residual Analysis - {best_name}', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'\nMean Residual: {residuals.mean():.4f}')
print(f'Residual Std: {residuals.std():.4f}')
print(f'Max |Residual|: {residuals.abs().max():.4f}')

In [ ]:
# ===== VERIFY: Test with sample input =====

hours, prev, extra, sleep, papers = 8, 85, 1, 7, 5

sample = pd.DataFrame({
    'Hours Studied': [hours],
    'Previous Scores': [prev],
    'Extracurricular Activities': [extra],
    'Sleep Hours': [sleep],
    'Sample Question Papers Practiced': [papers],
    'Study_Sleep_Ratio': [hours / (sleep + 1)],
    'Total_Effort': [hours + papers],
    'Previous_Score_Normalized': [prev / 100],
    'Sleep_Efficiency': [sleep * prev / 100],
    'Study_Score_Interaction': [hours * prev / 100],
    'Effort_Efficiency': [(hours + papers) * (prev / 100)],
    'Study_Squared': [hours ** 2],
    'Score_Squared': [(prev / 100) ** 2],
    'Papers_Per_Hour': [papers / (hours + 1)],
    'Balanced_Lifestyle': [sleep * prev / 100 * (extra if extra else 0.5)],
})

pred = best_model.predict(sample)[0]
print(f'Test Prediction: {pred:.2f} / 100')
print(f'Model: {best_name}')
print(f'Input: {hours}h study, {prev}% prev score, extra={bool(extra)}, {sleep}h sleep, {papers} papers')